Boundary Flux Jacobian 

Function 
- ghost_state
- boundary_flux
- boundary_flux_jacobian (by finite difference)

Boundary Conditions to implement
- slip wall/ symmetric BC
- noslip adiabatic wall
- riemann invariant (subsonic inflow and outflow, supersonic inflow and outflow)

In [1]:
import jax
import jax.numpy as jnp
import numpy as np
import sympy as sp

jax.config.update("jax_enable_x64", True)   # use float64


# ghost state function with slip wall and symmetric BC only 

def ghost_state(U, bc_type, n):
    """
    Compute ghost cell state for compressible N-S boundary conditions.

    Parameters
    ----------
    U       : array-like, shape (5,)
              Conservative variables [rho, rho*u, rho*v, rho*w, rho*E]
    bc_type : str
              'slip' for slip wall / symmetry, 'noslip' for no-slip wall
    n       : array-like, shape (3,)
              Outward unit normal vector [nx, ny, nz]

    Returns
    -------
    U_ghost : ndarray, shape (5,)
              Ghost cell conservative variables
    """
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)  # ensure unit normal

    rho = U[0]
    rho_vel = U[1:4]          # [rho*u, rho*v, rho*w]
    rho_E = U[4]

    vel = rho_vel / rho        # [u, v, w]

    U_ghost = np.empty(5)

    if bc_type == 'slip':
        # Reflect normal velocity component, keep tangential
        # v_ghost = v - 2*(v·n)*n
        vn = np.dot(vel, n)
        vel_ghost = vel - 2.0 * vn * n

        U_ghost[0] = rho
        U_ghost[1:4] = rho * vel_ghost
        U_ghost[4] = rho_E   # same total energy (pressure unchanged)

    elif bc_type == 'noslip':
        # Zero velocity at wall: ghost mirrors interior velocity
        # so that wall average = 0
        vel_ghost = -vel

        U_ghost[0] = rho
        U_ghost[1:4] = rho * vel_ghost
        U_ghost[4] = rho_E   # same total energy (isothermal assumption)

    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'. Use 'slip' or 'noslip'.")

    return U_ghost

Verification of the ghost-state function with slip-wall, no-slip wall conditions
- Check the fluxes
- For slip wall (at the boundary) --> scalar conserved, normal-velocity = 0
- For no-slip wall (at the boundary) --> scalar conserved, normal/tagential velocity = 0

In [2]:
def central_wall_state(U_interior, U_ghost):
    """Central scheme: wall value is average of interior and ghost."""
    return 0.5 * (U_interior + U_ghost)


def run_tests():
    tol = 1e-12
    passed = 0
    failed = 0

    def check(name, condition, details=""):
        nonlocal passed, failed
        if condition:
            print(f"  PASS  {name}")
            passed += 1
        else:
            print(f"  FAIL  {name}  {details}")
            failed += 1

    # ------------------------------------------------------------------ #
    # Test cases
    # Each entry: (label, U, bc_type, normal)
    # ------------------------------------------------------------------ #
    cases = [
        # 1. Slip wall, normal in x — only u should be zeroed at wall
        ("slip / n=x / flow diagonal",
         np.array([1.2, 1.2*3.0, 1.2*2.0, 1.2*1.0, 300.0]),
         'slip', np.array([1.0, 0.0, 0.0])),

        # 2. Slip wall, normal in y
        ("slip / n=y / flow diagonal",
         np.array([1.0, 1.0*1.5, 1.0*4.0, 1.0*(-2.0), 250.0]),
         'slip', np.array([0.0, 1.0, 0.0])),

        # 3. Slip wall, oblique normal (45 deg in x-y plane)
        ("slip / n=oblique(xy) / flow arbitrary",
         np.array([0.8, 0.8*2.0, 0.8*(-1.0), 0.8*3.0, 180.0]),
         'slip', np.array([1.0, 1.0, 0.0])),

        # 4. Slip wall, flow purely tangential (vn=0 already)
        ("slip / n=x / flow purely tangential",
         np.array([1.0, 0.0, 1.0*5.0, 1.0*(-3.0), 200.0]),
         'slip', np.array([1.0, 0.0, 0.0])),

        # 5. No-slip wall, normal in z
        ("noslip / n=z / general flow",
         np.array([1.5, 1.5*2.0, 1.5*(-1.0), 1.5*3.0, 400.0]),
         'noslip', np.array([0.0, 0.0, 1.0])),

        # 6. No-slip wall, oblique normal
        ("noslip / n=oblique(xyz) / general flow",
         np.array([1.1, 1.1*1.0, 1.1*2.0, 1.1*3.0, 350.0]),
         'noslip', np.array([1.0, 1.0, 1.0])),
    ]

    # ------------------------------------------------------------------ #
    for label, U, bc_type, n_raw in cases:
        print(f"\n[{bc_type.upper()}] {label}")
        n = n_raw / np.linalg.norm(n_raw)
        U_ghost = ghost_state(U, bc_type, n)
        U_wall  = central_wall_state(U, U_ghost)

        rho_wall = U_wall[0]
        vel_wall = U_wall[1:4] / rho_wall
        rho_int  = U[0]
        vel_int  = U[1:4] / rho_int

        if bc_type == 'slip':
            vn_wall = np.dot(vel_wall, n)
            vt_int  = vel_int - np.dot(vel_int, n) * n   # tangential of interior
            vt_wall = vel_wall - np.dot(vel_wall, n) * n

            check("wall normal velocity = 0",
                  abs(vn_wall) < tol,
                  f"vn_wall = {vn_wall:.3e}")
            check("wall tangential velocity preserved",
                  np.linalg.norm(vt_wall - vt_int) < tol,
                  f"Δvt = {np.linalg.norm(vt_wall - vt_int):.3e}")
            check("wall density = interior density",
                  abs(rho_wall - rho_int) < tol,
                  f"rho_wall={rho_wall:.4f}, rho_int={rho_int:.4f}")
            check("wall energy = interior energy",
                  abs(U_wall[4] - U[4]) < tol,
                  f"E_wall={U_wall[4]:.4f}, E_int={U[4]:.4f}")

        elif bc_type == 'noslip':
            check("wall velocity = 0 (all components)",
                  np.linalg.norm(vel_wall) < tol,
                  f"|v_wall| = {np.linalg.norm(vel_wall):.3e}")
            check("wall density = interior density",
                  abs(rho_wall - rho_int) < tol,
                  f"rho_wall={rho_wall:.4f}, rho_int={rho_int:.4f}")
            check("wall energy = interior energy",
                  abs(U_wall[4] - U[4]) < tol,
                  f"E_wall={U_wall[4]:.4f}, E_int={U[4]:.4f}")

    # ------------------------------------------------------------------ #
    print(f"\n{'='*45}")
    print(f"Results: {passed} passed, {failed} failed out of {passed+failed} checks")

run_tests()


[SLIP] slip / n=x / flow diagonal
  PASS  wall normal velocity = 0
  PASS  wall tangential velocity preserved
  PASS  wall density = interior density
  PASS  wall energy = interior energy

[SLIP] slip / n=y / flow diagonal
  PASS  wall normal velocity = 0
  PASS  wall tangential velocity preserved
  PASS  wall density = interior density
  PASS  wall energy = interior energy

[SLIP] slip / n=oblique(xy) / flow arbitrary
  PASS  wall normal velocity = 0
  PASS  wall tangential velocity preserved
  PASS  wall density = interior density
  PASS  wall energy = interior energy

[SLIP] slip / n=x / flow purely tangential
  PASS  wall normal velocity = 0
  PASS  wall tangential velocity preserved
  PASS  wall density = interior density
  PASS  wall energy = interior energy

[NOSLIP] noslip / n=z / general flow
  PASS  wall velocity = 0 (all components)
  PASS  wall density = interior density
  PASS  wall energy = interior energy

[NOSLIP] noslip / n=oblique(xyz) / general flow
  PASS  wall vel

riemann_invariant_bc
- return ghost cell for a riemann invariant boundary condition
- determine normal Mach number and flow direction with Mn_i = Vn_i / c_i
- extrapolate U_ghost = 2.0 * U_wall_c - U_int

In [3]:
def riemann_invariant_bc(U_int, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas=287.0):
    """
    Ghost cell state using Riemann invariant (characteristic) boundary condition.

    Incoming/outgoing waves are split based on sign of eigenvalues along n.
    Outgoing invariants are taken from the interior; incoming from prescribed state.

    Parameters
    ----------
    U_int  : array (5,)   interior conservative state [rho, rho*u, rho*v, rho*w, rho*E]
    n      : array (3,)   outward unit normal
    gamma  : float        ratio of specific heats
    u_b, v_b, w_b : float prescribed velocity components at boundary
    T_b    : float        prescribed temperature at boundary
    P_b    : float        prescribed pressure at boundary
    R_gas  : float        specific gas constant (default: air, 287 J/kg/K)

    Returns
    -------
    U_ghost : ndarray (5,)  ghost cell conservative variables
    info    : dict          diagnostic info (flow regime, Riemann invariants)
    """
    U_int = np.asarray(U_int, dtype=float)
    n     = np.asarray(n,     dtype=float)
    n     = n / np.linalg.norm(n)

    cv = R_gas / (gamma - 1.0)

    # ------------------------------------------------------------------ #
    # Interior primitives
    # ------------------------------------------------------------------ #
    rho_i = U_int[0]
    vel_i = U_int[1:4] / rho_i
    E_i   = U_int[4] / rho_i                         # specific total energy
    Vn_i  = np.dot(vel_i, n)                          # normal velocity (signed)
    Vt_i  = vel_i - Vn_i * n                          # tangential velocity vector
    p_i   = (gamma - 1.0) * rho_i * (E_i - 0.5 * np.dot(vel_i, vel_i))
    T_i   = p_i / (rho_i * R_gas)
    c_i   = np.sqrt(gamma * R_gas * T_i)

    # ------------------------------------------------------------------ #
    # Prescribed (boundary) primitives
    # ------------------------------------------------------------------ #
    rho_b = P_b / (R_gas * T_b)
    vel_b = np.array([u_b, v_b, w_b])
    Vn_b  = np.dot(vel_b, n)
    Vt_b  = vel_b - Vn_b * n
    c_b   = np.sqrt(gamma * R_gas * T_b)

    # ------------------------------------------------------------------ #
    # Riemann invariants along normal direction
    #   R+ = Vn + 2c/(gamma-1)   associated with lambda = Vn + c
    #   R- = Vn - 2c/(gamma-1)   associated with lambda = Vn - c
    # ------------------------------------------------------------------ #
    fac = 2.0 / (gamma - 1.0)
    Rp_int = Vn_i + fac * c_i    # R+ from interior
    Rm_int = Vn_i - fac * c_i    # R- from interior
    Rp_b   = Vn_b + fac * c_b    # R+ from prescribed
    Rm_b   = Vn_b - fac * c_b    # R- from prescribed

    Mn_i = Vn_i / c_i            # normal Mach number (interior)

    # ------------------------------------------------------------------ #
    # Select invariants based on flow regime
    # ------------------------------------------------------------------ #
    if Mn_i <= -1.0:
        # Supersonic inflow: all characteristics incoming → use prescribed
        regime   = "supersonic_inflow"
        Vn_wall  = Vn_b
        c_wall   = c_b
        Vt_wall  = Vt_b
        s_wall   = P_b / (rho_b ** gamma)             # entropy from prescribed

    elif Mn_i >= 1.0:
        # Supersonic outflow: all characteristics outgoing → use interior
        regime   = "supersonic_outflow"
        Vn_wall  = Vn_i
        c_wall   = c_i
        Vt_wall  = Vt_i
        s_wall   = p_i / (rho_i ** gamma)

    elif Vn_i < 0.0:
        # Subsonic inflow: R+ outgoing (from interior), R- incoming (from prescribed)

        # R- incoming (from farfield to domain) --> use prescribed value from farfield/ inlet
        # tagential velocity and entropy use prescribed value from farfield
        # R+ from interior

        regime   = "subsonic_inflow"
        Rp_use   = Rp_int
        Rm_use   = Rm_b
        Vn_wall  = 0.5 * (Rp_use + Rm_use)
        c_wall   = 0.25 * (gamma - 1.0) * (Rp_use - Rm_use)
        Vt_wall  = Vt_b                               # tangential from prescribed
        s_wall   = P_b / (rho_b ** gamma)             # entropy from prescribed

    else:
        # Subsonic outflow: R- incoming (from prescribed), rest from interior
        # R- incoming (from farfield to domain) --> use prescribed value from farfield/ outlet
        # tagential velocity and entropy use prescribed value from interior/ domain
        # R+ from interior

        regime   = "subsonic_outflow"
        Rp_use   = Rp_int
        Rm_use   = Rm_b
        Vn_wall  = 0.5 * (Rp_use + Rm_use)
        c_wall   = 0.25 * (gamma - 1.0) * (Rp_use - Rm_use)
        Vt_wall  = Vt_i                               # tangential from interior
        s_wall   = p_i / (rho_i ** gamma)             # entropy from interior

    # ------------------------------------------------------------------ #
    # Reconstruct wall primitive state from (Vn, c, Vt, entropy)
    # ------------------------------------------------------------------ #
    if c_wall <= 0.0:
        raise ValueError(f"Non-physical c_wall={c_wall:.4f} in regime '{regime}'.")

    rho_wall = (c_wall**2 / (gamma * s_wall)) ** (1.0 / (gamma - 1.0))
    p_wall   = s_wall * rho_wall ** gamma
    T_wall   = p_wall / (rho_wall * R_gas)
    vel_wall = Vn_wall * n + Vt_wall

    # ------------------------------------------------------------------ #
    # Ghost cell: reflect wall state through interior so that
    # central average (U_int + U_ghost)/2 = U_wall
    # ------------------------------------------------------------------ #
    E_wall    = cv * T_wall + 0.5 * np.dot(vel_wall, vel_wall)
    U_wall_c  = np.array([
        rho_wall,
        rho_wall * vel_wall[0],
        rho_wall * vel_wall[1],
        rho_wall * vel_wall[2],
        rho_wall * E_wall,
    ])
    U_ghost = 2.0 * U_wall_c - U_int

    info = {
        "regime"  : regime,
        "Mn_i"    : Mn_i,
        "Rp_int"  : Rp_int, "Rm_int": Rm_int,
        "Rp_b"    : Rp_b,   "Rm_b"  : Rm_b,
        "Vn_wall" : Vn_wall, "c_wall": c_wall,
        "p_wall"  : p_wall,  "T_wall": T_wall,
    }
    return U_ghost, info

In [4]:
# test case to verify riemann_invariant_bc

def central_wall(U_int, U_ghost):
    return 0.5 * (U_int + U_ghost)

def primitives(U, gamma, R_gas=287.0):
    """Unpack conservative → (rho, vel, p, T, c)."""
    rho = U[0]
    vel = U[1:4] / rho
    cv  = R_gas / (gamma - 1.0)
    E   = U[4] / rho
    p   = (gamma - 1.0) * rho * (E - 0.5 * np.dot(vel, vel))
    T   = p / (rho * R_gas)
    c   = np.sqrt(gamma * R_gas * T)
    return rho, vel, p, T, c

def make_U(rho, u, v, w, T, gamma, R_gas=287.0):
    cv  = R_gas / (gamma - 1.0)
    E   = cv * T + 0.5 * (u**2 + v**2 + w**2)
    return np.array([rho, rho*u, rho*v, rho*w, rho*E])

def run_tests():
    gamma = 1.4
    R_gas = 287.0
    tol   = 1e-10
    passed = failed = 0

    def check(name, val, ref, atol=tol, note=""):
        nonlocal passed, failed
        err = abs(val - ref)
        ok  = err < atol
        tag = "PASS" if ok else "FAIL"
        detail = f"  got={val:.6e}  ref={ref:.6e}  err={err:.2e}"
        print(f"  {tag}  {name}{detail}  {note}")
        if ok: passed += 1
        else:  failed += 1

    fac = 2.0 / (gamma - 1.0)   # 2/(γ-1)

    # ================================================================== #
    # CASE 1 — Subsonic INFLOW (Vn < 0, |Mn| < 1)
    #
    # Verify:
    #   (a) R+ at wall == R+ from interior
    #   (b) R- at wall == R- from prescribed
    #   (c) tangential velocity at wall == prescribed tangential
    #   (d) entropy at wall == prescribed entropy
    # ================================================================== #
    print("\n" + "="*60)
    print("CASE 1 — Subsonic inflow")

    n    = np.array([1.0, 0.0, 0.0])
    # interior: flow moving toward boundary (Vn < 0)
    U_i  = make_U(rho=1.2, u=-120.0, v=50.0, w=20.0, T=300.0, gamma=gamma)
    # prescribed farfield (inflow state)
    u_b, v_b, w_b, T_b, P_b = -150.0, 30.0, 10.0, 280.0, 90000.0

    U_g, info = riemann_invariant_bc(U_i, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)
    U_w       = central_wall(U_i, U_g)

    rho_i, vel_i, p_i, T_i, c_i = primitives(U_i,  gamma)
    rho_w, vel_w, p_w, T_w, c_w = primitives(U_w, gamma)

    Vn_i = np.dot(vel_i, n);  Vn_w = np.dot(vel_w, n)
    Vt_i = vel_i - Vn_i*n;   Vt_w = vel_w - Vn_w*n

    rho_b = P_b / (R_gas * T_b)
    c_b   = np.sqrt(gamma * R_gas * T_b)
    Vn_b  = u_b   # since n = [1,0,0]
    vel_b = np.array([u_b, v_b, w_b])
    Vt_b  = vel_b - Vn_b*n

    Rp_int = Vn_i + fac*c_i
    Rm_b   = Vn_b - fac*c_b
    Rp_w   = Vn_w + fac*c_w
    Rm_w   = Vn_w - fac*c_w
    s_b    = P_b / rho_b**gamma
    s_w    = p_w / rho_w**gamma

    print(f"  regime = {info['regime']}")
    check("R+ wall == R+ interior",    Rp_w, Rp_int)
    check("R- wall == R- prescribed",  Rm_w, Rm_b)
    check("Vt_y wall == Vt_y prescr.", Vt_w[1], Vt_b[1])
    check("Vt_z wall == Vt_z prescr.", Vt_w[2], Vt_b[2])
    check("entropy wall == entropy prescribed", s_w, s_b, atol=1e-6)

    # ================================================================== #
    # CASE 2 — Subsonic OUTFLOW (Vn > 0, |Mn| < 1)
    #
    # Verify:
    #   (a) R+ at wall == R+ from interior
    #   (b) R- at wall == R- from prescribed
    #   (c) tangential velocity at wall == interior tangential
    #   (d) entropy at wall == interior entropy
    # ================================================================== #
    print("\n" + "="*60)
    print("CASE 2 — Subsonic outflow")

    n    = np.array([0.0, 1.0, 0.0])
    U_i  = make_U(rho=1.0, u=30.0, v=100.0, w=-20.0, T=320.0, gamma=gamma)
    u_b, v_b, w_b, T_b, P_b = 10.0, 80.0, 0.0, 310.0, 95000.0

    U_g, info = riemann_invariant_bc(U_i, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)
    U_w       = central_wall(U_i, U_g)

    rho_i, vel_i, p_i, T_i, c_i = primitives(U_i,  gamma)
    rho_w, vel_w, p_w, T_w, c_w = primitives(U_w, gamma)

    Vn_i = np.dot(vel_i, n);  Vn_w = np.dot(vel_w, n)
    Vt_i = vel_i - Vn_i*n;   Vt_w = vel_w - Vn_w*n

    rho_b = P_b / (R_gas * T_b)
    c_b   = np.sqrt(gamma * R_gas * T_b)
    Vn_b  = v_b
    vel_b = np.array([u_b, v_b, w_b])
    Vt_b  = vel_b - Vn_b*n

    Rp_int = Vn_i + fac*c_i
    Rm_b   = Vn_b - fac*c_b
    Rp_w   = Vn_w + fac*c_w
    Rm_w   = Vn_w - fac*c_w
    s_i    = p_i / rho_i**gamma
    s_w    = p_w / rho_w**gamma

    print(f"  regime = {info['regime']}")
    check("R+ wall == R+ interior",    Rp_w, Rp_int)
    check("R- wall == R- prescribed",  Rm_w, Rm_b)
    check("Vt_x wall == Vt_x interior", Vt_w[0], Vt_i[0])
    check("Vt_z wall == Vt_z interior", Vt_w[2], Vt_i[2])
    check("entropy wall == entropy interior", s_w, s_i, atol=1e-6)

    # ================================================================== #
    # CASE 3 — Supersonic INFLOW (Vn < 0, |Mn| > 1)
    #
    # All characteristics incoming → wall state == prescribed state entirely
    # ================================================================== #
    print("\n" + "="*60)
    print("CASE 3 — Supersonic inflow")

    n    = np.array([1.0, 0.0, 0.0])
    # c ≈ 347 m/s, so Vn = -800 → Mn ≈ -2.3
    U_i  = make_U(rho=0.5, u=-800.0, v=50.0, w=0.0, T=300.0, gamma=gamma)
    u_b, v_b, w_b, T_b, P_b = -820.0, 40.0, 5.0, 295.0, 45000.0

    U_g, info = riemann_invariant_bc(U_i, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)
    U_w       = central_wall(U_i, U_g)

    rho_b_ref = P_b / (R_gas * T_b)
    rho_w, vel_w, p_w, T_w, c_w = primitives(U_w, gamma)

    print(f"  regime = {info['regime']}")
    check("rho wall == rho prescribed",   rho_w,     rho_b_ref, atol=1e-8)
    check("u   wall == u   prescribed",   vel_w[0],  u_b,       atol=1e-8)
    check("v   wall == v   prescribed",   vel_w[1],  v_b,       atol=1e-8)
    check("p   wall == p   prescribed",   p_w,       P_b,       atol=1e-4)
    check("T   wall == T   prescribed",   T_w,       T_b,       atol=1e-8)

    # ================================================================== #
    # CASE 4 — Supersonic OUTFLOW (Vn > 0, |Mn| > 1)
    #
    # All characteristics outgoing → wall state == interior state entirely
    # ================================================================== #
    print("\n" + "="*60)
    print("CASE 4 — Supersonic outflow")

    n    = np.array([0.0, 0.0, 1.0])
    # c ≈ 347 m/s, Vn = 900 → Mn ≈ 2.6
    U_i  = make_U(rho=0.7, u=10.0, v=-20.0, w=900.0, T=300.0, gamma=gamma)
    u_b, v_b, w_b, T_b, P_b = 5.0, -15.0, 850.0, 290.0, 50000.0

    U_g, info = riemann_invariant_bc(U_i, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)
    U_w       = central_wall(U_i, U_g)

    rho_i, vel_i, p_i, T_i, c_i = primitives(U_i, gamma)
    rho_w, vel_w, p_w, T_w, c_w = primitives(U_w, gamma)

    print(f"  regime = {info['regime']}")
    check("rho wall == rho interior",  rho_w,    rho_i,    atol=1e-8)
    check("u   wall == u   interior",  vel_w[0], vel_i[0], atol=1e-8)
    check("v   wall == v   interior",  vel_w[1], vel_i[1], atol=1e-8)
    check("w   wall == w   interior",  vel_w[2], vel_i[2], atol=1e-8)
    check("p   wall == p   interior",  p_w,      p_i,      atol=1e-6)

    # ================================================================== #
    # CASE 5 — Oblique normal, subsonic outflow
    #
    # Ensures normal decomposition is correct when n is not axis-aligned
    # ================================================================== #
    print("\n" + "="*60)
    print("CASE 5 — Oblique normal (subsonic outflow)")

    n    = np.array([1.0, 1.0, 0.0]) / np.sqrt(2.0)
    U_i  = make_U(rho=1.1, u=120.0, v=100.0, w=30.0, T=310.0, gamma=gamma)
    u_b, v_b, w_b, T_b, P_b = 0.0, 0.0, 0.0, 300.0, 101325.0

    U_g, info = riemann_invariant_bc(U_i, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)
    U_w       = central_wall(U_i, U_g)

    rho_i, vel_i, p_i, T_i, c_i = primitives(U_i,  gamma)
    rho_w, vel_w, p_w, T_w, c_w = primitives(U_w, gamma)

    Vn_i = np.dot(vel_i, n);  Vn_w = np.dot(vel_w, n)
    Vt_i = vel_i - Vn_i*n;   Vt_w = vel_w - Vn_w*n

    rho_b = P_b / (R_gas * T_b)
    c_b   = np.sqrt(gamma * R_gas * T_b)
    vel_b = np.array([u_b, v_b, w_b])
    Vn_b  = np.dot(vel_b, n)

    Rp_int = Vn_i + fac*c_i
    Rm_b   = Vn_b - fac*c_b
    Rp_w   = Vn_w + fac*c_w
    Rm_w   = Vn_w - fac*c_w
    s_i    = p_i / rho_i**gamma
    s_w    = p_w / rho_w**gamma

    print(f"  regime = {info['regime']}")
    check("R+ wall == R+ interior",         Rp_w,     Rp_int)
    check("R- wall == R- prescribed",        Rm_w,     Rm_b)
    check("tangential vel preserved (norm)", np.linalg.norm(Vt_w - Vt_i), 0.0, atol=1e-8)
    check("entropy wall == entropy interior", s_w,     s_i,  atol=1e-6)

    # ================================================================== #
    print(f"\n{'='*60}")
    print(f"Results: {passed} passed, {failed} failed out of {passed+failed} checks")

run_tests()


CASE 1 — Subsonic inflow
  regime = subsonic_inflow
  PASS  R+ wall == R+ interior  got=1.615944e+03  ref=1.615944e+03  err=0.00e+00  
  PASS  R- wall == R- prescribed  got=-1.827081e+03  ref=-1.827081e+03  err=0.00e+00  
  PASS  Vt_y wall == Vt_y prescr.  got=3.000000e+01  ref=3.000000e+01  err=0.00e+00  
  PASS  Vt_z wall == Vt_z prescr.  got=1.000000e+01  ref=1.000000e+01  err=0.00e+00  
  PASS  entropy wall == entropy prescribed  got=7.679959e+04  ref=7.679959e+04  err=0.00e+00  

CASE 2 — Subsonic outflow
  regime = subsonic_outflow
  PASS  R+ wall == R+ interior  got=1.892875e+03  ref=1.892875e+03  err=4.55e-13  
  PASS  R- wall == R- prescribed  got=-1.684639e+03  ref=-1.684639e+03  err=6.82e-13  
  PASS  Vt_x wall == Vt_x interior  got=3.000000e+01  ref=3.000000e+01  err=0.00e+00  
  PASS  Vt_z wall == Vt_z interior  got=-2.000000e+01  ref=-2.000000e+01  err=0.00e+00  
  PASS  entropy wall == entropy interior  got=9.184000e+04  ref=9.184000e+04  err=1.46e-11  

CASE 3 — Supers

In [5]:
# ghost_state function with Riemann Invariant boundary conditions

def ghost_state(U, bc_type, n, gamma=1.4, R_gas=287.0,
                u_b=0.0, v_b=0.0, w_b=0.0, T_b=300.0, P_b=101325.0):
    """
    Compute ghost cell state for compressible N-S boundary conditions.

    Parameters
    ----------
    U       : array (5,)  conservative variables [rho, rho*u, rho*v, rho*w, rho*E]
    bc_type : str         'slip', 'noslip', or 'riemann'
    n       : array (3,)  outward unit normal

    For bc_type == 'riemann' only:
    gamma   : float       ratio of specific heats
    R_gas   : float       specific gas constant (J/kg/K)
    u_b, v_b, w_b : float prescribed boundary velocity
    T_b     : float       prescribed boundary temperature
    P_b     : float       prescribed boundary pressure

    Returns
    -------
    U_ghost : ndarray (5,)
    """
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)

    rho   = U[0]
    vel   = U[1:4] / rho
    rho_E = U[4]

    if bc_type == 'slip':
        vn        = np.dot(vel, n)
        vel_ghost = vel - 2.0 * vn * n
        return np.array([rho, *(rho * vel_ghost), rho_E])

    elif bc_type == 'noslip':
        return np.array([rho, *(-rho * vel), rho_E])

    elif bc_type == 'riemann':
        return riemann_invariant_bc(U, n, gamma, u_b, v_b, w_b, T_b, P_b, R_gas)[0]  # only return the first output, the second output is for debugging

    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'. Use 'slip', 'noslip', or 'riemann'.")

Verifications of ghost_state_function by checking Euler Flux

For slip wall/symmetric, U_wall = Ug + Ui, 
- F(U_wall)[0] and F(U_wall)[4] = 0, zero mass and energy flux
- F(U_wall)[1] = P * nx, only pressure flux



In [6]:
def normal_euler_flux(U, n, gamma=1.4):
    rho  = U[0]
    rhou, rhov, rhow, rhoE = U[1], U[2], U[3], U[4]
    u, v, w = rhou/rho, rhov/rho, rhow/rho
    nx, ny, nz = n[0], n[1], n[2]
    un = u*nx + v*ny + w*nz
    P  = (gamma - 1.0) * (rhoE - 0.5*rho*(u**2 + v**2 + w**2))
    F  = np.zeros(5)
    F[0] = rho  * un
    F[1] = rhou * un + P*nx
    F[2] = rhov * un + P*ny
    F[3] = rhow * un + P*nz
    F[4] = (rhoE + P) * un
    return F


def run_noslip_flux_test():
    gamma = 1.4
    R_gas = 287.0
    tol   = 1e-10
    passed = failed = 0

    def check(name, val, ref, atol=tol):
        nonlocal passed, failed
        err = abs(val - ref)
        ok  = err < atol
        print(f"  {'PASS' if ok else 'FAIL'}  {name}  val={val:.6e}  ref={ref:.6e}  err={err:.2e}")
        if ok: passed += 1
        else:  failed += 1

    def make_U(rho, u, v, w, T):
        cv = R_gas / (gamma - 1.0)
        E  = cv*T + 0.5*(u**2 + v**2 + w**2)
        return np.array([rho, rho*u, rho*v, rho*w, rho*E])

    cases = [
        ("n = x-axis",      np.array([1., 0., 0.]),             make_U(1.2,  80., 50.,  30., 300.)),
        ("n = y-axis",      np.array([0., 1., 0.]),             make_U(1.0,  20.,-60.,  10., 280.)),
        ("n = z-axis",      np.array([0., 0., 1.]),             make_U(1.3,  40., 30., 100., 320.)),
        ("n = oblique xy",  np.array([1., 1., 0.])/np.sqrt(2.), make_U(1.1, 100.,100.,  20., 310.)),
        ("n = oblique xyz", np.array([1., 1., 1.])/np.sqrt(3.), make_U(0.9,  60.,-40.,  80., 295.)),
    ]

    for label, n_raw, U_i in cases:
        n      = n_raw / np.linalg.norm(n_raw)
        U_g    = ghost_state(U_i, 'noslip', n)

        # evaluate flux AT the face state, not average of two fluxes
        U_wall = 0.5 * (U_i + U_g)
        F      = normal_euler_flux(U_wall, n, gamma)

        # wall pressure: velocity = 0 at wall, so P_wall = (γ-1)*rhoE_i
        P_wall = (gamma - 1.0) * U_i[4]

        print(f"\n[NOSLIP] {label}  P_wall = {P_wall:.2f} Pa")
        check("F[0] = 0           (mass flux)",   F[0], 0.0)
        check(f"F[1] = P*nx",                     F[1], P_wall * n[0])
        check(f"F[2] = P*ny",                     F[2], P_wall * n[1])
        check(f"F[3] = P*nz",                     F[3], P_wall * n[2])
        check("F[4] = 0           (energy flux)", F[4], 0.0)

    print(f"\n{'='*55}")
    print(f"Results: {passed} passed, {failed} failed out of {passed+failed} checks")

run_noslip_flux_test()


[NOSLIP] n = x-axis  P_wall = 105672.00 Pa
  PASS  F[0] = 0           (mass flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[1] = P*nx  val=1.056720e+05  ref=1.056720e+05  err=0.00e+00
  PASS  F[2] = P*ny  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[3] = P*nz  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[4] = 0           (energy flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00

[NOSLIP] n = y-axis  P_wall = 81180.00 Pa
  PASS  F[0] = 0           (mass flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[1] = P*nx  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[2] = P*ny  val=8.118000e+04  ref=8.118000e+04  err=0.00e+00
  PASS  F[3] = P*nz  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[4] = 0           (energy flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00

[NOSLIP] n = z-axis  P_wall = 122642.00 Pa
  PASS  F[0] = 0           (mass flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00


In [7]:
def run_all_bc_flux_tests():
    gamma = 1.4
    R_gas = 287.0
    tol   = 1e-10
    passed = failed = 0

    def check(name, val, ref, atol=tol):
        nonlocal passed, failed
        err = abs(val - ref)
        ok  = err < atol
        print(f"  {'PASS' if ok else 'FAIL'}  {name}  val={val:.6e}  ref={ref:.6e}  err={err:.2e}")
        if ok: passed += 1
        else:  failed += 1

    def make_U(rho, u, v, w, T):
        cv = R_gas / (gamma - 1.0)
        E  = cv*T + 0.5*(u**2 + v**2 + w**2)
        return np.array([rho, rho*u, rho*v, rho*w, rho*E])

    def face_flux(U_i, U_g, n):
        U_wall = 0.5 * (U_i + U_g)
        return normal_euler_flux(U_wall, n, gamma), U_wall

    def wall_pressure(U_wall):
        rho = U_wall[0]; vel = U_wall[1:4] / rho
        return (gamma - 1.0) * (U_wall[4] - 0.5 * rho * np.dot(vel, vel))

    fac = 2.0 / (gamma - 1.0)

    # ================================================================== #
    # SLIP WALL
    # U_wall has zero normal velocity, tangential preserved.
    # F(U_wall) = [0, P_wall*nx, P_wall*ny, P_wall*nz, 0]
    # P_wall = P_i + (γ-1)*0.5*rho_i*Vn_i²  (Vn removed → higher pressure)
    # ================================================================== #
    slip_cases = [
        ("slip / n=x",        np.array([1., 0., 0.]),             make_U(1.2,  80., 50.,  30., 300.)),
        ("slip / n=y",        np.array([0., 1., 0.]),             make_U(1.0,  20.,-60.,  10., 280.)),
        ("slip / n=oblique",  np.array([1., 1., 1.])/np.sqrt(3.), make_U(0.9,  60.,-40.,  80., 295.)),
    ]

    for label, n_raw, U_i in slip_cases:
        n          = n_raw / np.linalg.norm(n_raw)
        U_g        = ghost_state(U_i, 'slip', n)
        F, U_wall  = face_flux(U_i, U_g, n)
        P_wall     = wall_pressure(U_wall)

        print(f"\n[SLIP]   {label}  P_wall={P_wall:.2f}")
        check("F[0] = 0        (mass flux)",   F[0], 0.0)
        check("F[1] = P*nx",                   F[1], P_wall * n[0])
        check("F[2] = P*ny",                   F[2], P_wall * n[1])
        check("F[3] = P*nz",                   F[3], P_wall * n[2])
        check("F[4] = 0        (energy flux)", F[4], 0.0)

    # ================================================================== #
    # NO-SLIP WALL
    # U_wall has zero velocity entirely.
    # P_wall = (γ-1)*rhoE_i  (all kinetic energy → internal)
    # ================================================================== #
    noslip_cases = [
        ("noslip / n=x",       np.array([1., 0., 0.]),             make_U(1.2,  80., 50.,  30., 300.)),
        ("noslip / n=z",       np.array([0., 0., 1.]),             make_U(1.3,  40., 30., 100., 320.)),
        ("noslip / n=oblique", np.array([1., 1., 0.])/np.sqrt(2.), make_U(1.1, 100.,100.,  20., 310.)),
    ]

    for label, n_raw, U_i in noslip_cases:
        n          = n_raw / np.linalg.norm(n_raw)
        U_g        = ghost_state(U_i, 'noslip', n)
        F, U_wall  = face_flux(U_i, U_g, n)
        P_wall     = (gamma - 1.0) * U_i[4]   # all KE → internal energy at wall

        print(f"\n[NOSLIP] {label}  P_wall={P_wall:.2f}")
        check("F[0] = 0        (mass flux)",   F[0], 0.0)
        check("F[1] = P*nx",                   F[1], P_wall * n[0])
        check("F[2] = P*ny",                   F[2], P_wall * n[1])
        check("F[3] = P*nz",                   F[3], P_wall * n[2])
        check("F[4] = 0        (energy flux)", F[4], 0.0)

    # ================================================================== #
    # RIEMANN INVARIANT
    # No zero-flux condition — instead verify the face state satisfies
    # the correct Riemann invariants and entropy source.
    #
    # subsonic inflow:  R+ from interior, R- from prescribed, s from prescribed
    # subsonic outflow: R+ from interior, R- from prescribed, s from interior
    # supersonic inflow:  face state == prescribed
    # supersonic outflow: face state == interior
    # ================================================================== #
    riemann_cases = [
        ("riemann / subsonic inflow",
         np.array([1., 0., 0.]),
         make_U(1.2, -120., 50., 20., 300.),
         dict(u_b=-150., v_b=30., w_b=10., T_b=280., P_b=90000.)),

        ("riemann / subsonic outflow",
         np.array([0., 1., 0.]),
         make_U(1.0,  30., 100., -20., 320.),
         dict(u_b=10., v_b=80., w_b=0., T_b=310., P_b=95000.)),

        ("riemann / supersonic inflow",
         np.array([1., 0., 0.]),
         make_U(0.5, -800., 50., 0., 300.),
         dict(u_b=-820., v_b=40., w_b=5., T_b=295., P_b=45000.)),

        ("riemann / supersonic outflow",
         np.array([0., 0., 1.]),
         make_U(0.7,  10., -20., 900., 300.),
         dict(u_b=5., v_b=-15., w_b=850., T_b=290., P_b=50000.)),
    ]

    for label, n_raw, U_i, bc in riemann_cases:
        n         = n_raw / np.linalg.norm(n_raw)
        U_g       = ghost_state(U_i, 'riemann', n, gamma, R_gas, **bc)
        _, U_wall = face_flux(U_i, U_g, n)

        # face primitives
        rho_w = U_wall[0];  vel_w = U_wall[1:4] / rho_w
        P_w   = wall_pressure(U_wall)
        T_w   = P_w / (rho_w * R_gas)
        c_w   = np.sqrt(gamma * R_gas * T_w)
        Vn_w  = np.dot(vel_w, n)
        Vt_w  = vel_w - Vn_w * n

        # interior primitives
        rho_i = U_i[0];  vel_i = U_i[1:4] / rho_i
        P_i   = wall_pressure(U_i)
        T_i   = P_i / (rho_i * R_gas)
        c_i   = np.sqrt(gamma * R_gas * T_i)
        Vn_i  = np.dot(vel_i, n)
        Mn_i  = Vn_i / c_i

        # prescribed primitives
        rho_b = bc['P_b'] / (R_gas * bc['T_b'])
        c_b   = np.sqrt(gamma * R_gas * bc['T_b'])
        vel_b = np.array([bc['u_b'], bc['v_b'], bc['w_b']])
        Vn_b  = np.dot(vel_b, n)
        Vt_b  = vel_b - Vn_b * n

        Rp_int = Vn_i + fac * c_i;  Rm_b = Vn_b - fac * c_b
        Rp_w   = Vn_w + fac * c_w;  Rm_w = Vn_w - fac * c_w
        s_i    = P_i   / rho_i ** gamma
        s_b    = bc['P_b'] / rho_b ** gamma
        s_w    = P_w   / rho_w ** gamma

        print(f"\n[RIEMANN] {label}  Mn_i={Mn_i:.3f}")

        if Mn_i <= -1.0:                        # supersonic inflow
            check("face Vn  == prescribed Vn",  Vn_w,  Vn_b)
            check("face c   == prescribed c",   c_w,   c_b)
            check("face s   == prescribed s",   s_w,   s_b,  atol=1e-6)
            check("face Vt  == prescribed Vt",  np.linalg.norm(Vt_w - Vt_b), 0.0, atol=1e-8)

        elif Mn_i >= 1.0:                       # supersonic outflow
            check("face Vn  == interior Vn",    Vn_w,  Vn_i)
            check("face c   == interior c",     c_w,   c_i)
            check("face s   == interior s",     s_w,   s_i,  atol=1e-6)
            check("face Vt  == interior Vt",    np.linalg.norm(Vt_w - (vel_i - Vn_i*n)), 0.0, atol=1e-8)

        elif Vn_i < 0.0:                        # subsonic inflow
            check("R+ face == R+ interior",     Rp_w,  Rp_int)
            check("R- face == R- prescribed",   Rm_w,  Rm_b)
            check("entropy == prescribed",      s_w,   s_b,  atol=1e-6)
            check("face Vt == prescribed Vt",   np.linalg.norm(Vt_w - Vt_b), 0.0, atol=1e-8)

        else:                                   # subsonic outflow
            check("R+ face == R+ interior",     Rp_w,  Rp_int)
            check("R- face == R- prescribed",   Rm_w,  Rm_b)
            check("entropy == interior",        s_w,   s_i,  atol=1e-6)
            check("face Vt == interior Vt",     np.linalg.norm(Vt_w - (vel_i - Vn_i*n)), 0.0, atol=1e-8)

    print(f"\n{'='*55}")
    print(f"Results: {passed} passed, {failed} failed out of {passed+failed} checks")

run_all_bc_flux_tests()



[SLIP]   slip / n=x  P_wall=104856.00
  PASS  F[0] = 0        (mass flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[1] = P*nx  val=1.048560e+05  ref=1.048560e+05  err=0.00e+00
  PASS  F[2] = P*ny  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[3] = P*nz  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[4] = 0        (energy flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00

[SLIP]   slip / n=y  P_wall=81080.00
  PASS  F[0] = 0        (mass flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[1] = P*nx  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[2] = P*ny  val=8.108000e+04  ref=8.108000e+04  err=0.00e+00
  PASS  F[3] = P*nz  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00
  PASS  F[4] = 0        (energy flux)  val=0.000000e+00  ref=0.000000e+00  err=0.00e+00

[SLIP]   slip / n=oblique  P_wall=76798.50
  PASS  F[0] = 0        (mass flux)  val=-9.592327e-15  ref=0.000000e+00  err=9.59e-15
  PASS  F[1] = P*nx  val

Implement the boundary flux jacobian function
- for slip wall/ symmetric/ Riemann Invariant condition --> use inviscid_jacobian function
- for no-slip wall --> use vicous_jacobian function (neglect inviscid flux)

Utility functions
- inviscid_flux_jacobians and its dependents
- viscous_flux_jacobians and its dependents

In [8]:
# utility functions

# inviscid jacobian function
def compute_dFdU(U, normal, gamma=1.4):
    """
    Compute the inviscid flux Jacobian matrix for 3D Euler equations.

    Based on equation (A.47) from Chung's Computationa Fluid Dynamics.
    Parameters
    ----------
    U : array-like, shape (5,)
        Conservative variable vector [rho, rho*u, rho*v, rho*w, rho*E]
        where:
        - rho: density
        - u, v, w: velocity components
        - E: specific total energy

    normal : array-like, shape (3,)
        Unit normal vector [n_x, n_y, n_z]

    gamma : float, optional
        Specific heat ratio (default: 1.4 for air)

    Returns
    -------
    dF_dU : ndarray, shape (5, 5)
        Flux Jacobian matrix
    """

    # Extract conservative variables
    rho = U[0]
    rho_u = U[1]
    rho_v = U[2]
    rho_w = U[3]
    rho_E = U[4]

    # Extract normal vector components
    n_x = normal[0]
    n_y = normal[1]
    n_z = normal[2]

    # Compute primitive variables
    u = rho_u / rho
    v = rho_v / rho
    w = rho_w / rho
    E = rho_E / rho

    # Compute auxiliary quantities (from A.48)
    # phi = 0.5 * (gamma - 1) * (u^2 + v^2 + w^2)
    phi = 0.5 * (gamma - 1.0) * (u**2 + v**2 + w**2)

    # V = n_x*u + n_y*v + n_z*w (normal component of velocity)
    V = n_x * u + n_y * v + n_z * w

    # a1 = gamma*E - phi
    a1 = gamma * E - phi

    # a2 = gamma - 1
    a2 = gamma - 1.0

    # a3 = gamma - 2
    a3 = gamma - 2.0

    # Initialize Jacobian matrix
    dF_dU = np.zeros((5, 5))

    # Row 1: d(rho*V)/d(U)
    dF_dU[0, 0] = 0
    dF_dU[0, 1] = n_x
    dF_dU[0, 2] = n_y
    dF_dU[0, 3] = n_z
    dF_dU[0, 4] = 0.0

    # Row 2: d(rho*u*V)/d(U)
    dF_dU[1, 0] = n_x * phi - u * V
    dF_dU[1, 1] = V - a3 * n_x * u
    dF_dU[1, 2] = n_y * u - a2 * n_x *v
    dF_dU[1, 3] = n_z * u - a2 * n_x * w
    dF_dU[1, 4] = a2 * n_x

    # Row 3: d(rho*v*V)/d(U)
    dF_dU[2, 0] = n_y * phi - v * V
    dF_dU[2, 1] = n_x * v - a2 * n_y *u
    dF_dU[2, 2] = V - a3 * n_y * v
    dF_dU[2, 3] = n_z * v - a2 * n_y * w
    dF_dU[2, 4] = a2 * n_y

    # Row 4: d(rho*w*V)/d(U)
    dF_dU[3, 0] = n_z * phi - w * V
    dF_dU[3, 1] = n_x * w - a2 * n_z *u
    dF_dU[3, 2] = n_y * w - a2 * n_z *v
    dF_dU[3, 3] =  V - a3 * n_z * w
    dF_dU[3, 4] = a2 * n_z

    # Row 5: d(rho*H*V)/d(U)
    dF_dU[4, 0] = V * (phi - a1)
    dF_dU[4, 1] = a1 * n_x - a2 * u * V
    dF_dU[4, 2] = a1 * n_y - a2 * v * V
    dF_dU[4, 3] = a1 * n_z - a2 * w * V
    dF_dU[4, 4] = gamma * V

    return dF_dU


# compute_roe_averaged_absolute_jacobian





# function to output roe-averages

def roe_average(UL, UR, gamma=1.4):
    """
    Compute the Roe-averaged conservative variable vector for the 3D Euler equations.
 
    Parameters
    ----------
    UL, UR : array-like, shape (5,)
        Left/right conservative vectors [rho, rho*u, rho*v, rho*w, rho*E].
    gamma : float
        Ratio of specific heats (default 1.4).
 
    Returns
    -------
    U_roe : ndarray, shape (5,)
        Roe-averaged conservative variable vector.
    """
    UL = np.asarray(UL, dtype=float)
    UR = np.asarray(UR, dtype=float)
 
    rhoL, rhoR = UL[0], UR[0]
    velL = UL[1:4] / rhoL
    velR = UR[1:4] / rhoR
    EL   = UL[4] / rhoL
    ER   = UR[4] / rhoR
 
    pL = (gamma - 1.0) * rhoL * (EL - 0.5 * np.dot(velL, velL))
    pR = (gamma - 1.0) * rhoR * (ER - 0.5 * np.dot(velR, velR))
    HL = EL + pL / rhoL    # total enthalpy per unit mass
    HR = ER + pR / rhoR
 
    wL = np.sqrt(rhoL)     # Roe weight for left state
    wR = np.sqrt(rhoR)     # Roe weight for right state
    ws = wL + wR
 
    rho_roe = wL * wR                          # = sqrt(rhoL * rhoR)
    vel_roe = (wL * velL + wR * velR) / ws
    H_roe   = (wL * HL   + wR * HR)   / ws
 
    # Recover E from H:  H = gamma*E - (gamma-1)/2 * |v|^2
    q2_roe = np.dot(vel_roe, vel_roe)
    E_roe  = (H_roe + (gamma - 1.0) * 0.5 * q2_roe) / gamma
 
    U_roe = np.empty(5)
    U_roe[0]   = rho_roe
    U_roe[1:4] = rho_roe * vel_roe
    U_roe[4]   = rho_roe * E_roe
    
    return U_roe

"""
Absolute Normal Jacobian |A_n| for the 3D Euler equations.

Reference: equations 3.6.16 – 3.6.26.

Conservative variable vector:
    U = [rho, rho*u, rho*v, rho*w, rho*E]   (shape 5)

The result is assembled from three contributions (eqs. 3.6.24 – 3.6.26):

    |A_n| = |qn - c| * r1*l1^T          (eq. 3.6.25)
          + |qn|     * (r2*l2^T + r4*l4^T + r5*l5^T)   (eq. 3.6.24)
          + |qn + c| * r3*l3^T          (eq. 3.6.26)
"""

import numpy as np


def absolute_normal_jacobian(U, n, gamma=1.4):
    """
    Compute the absolute value of the normal Jacobian |A_n| for the
    3D Euler equations without tangent-vector ambiguity (eqs. 3.6.16–3.6.26).

    Parameters
    ----------
    U : array-like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E].
    n : array-like, shape (3,)
        Face normal vector (need not be unit length; normalised internally).
    gamma : float
        Ratio of specific heats (default 1.4).

    Returns
    -------
    abs_An : ndarray, shape (5, 5)
        Absolute normal Jacobian matrix.
    """
    U = np.asarray(U, dtype=float)
    n = np.asarray(n, dtype=float)
    n = n / np.linalg.norm(n)          # unit normal

    # ------------------------------------------------------------------ #
    # Primitive / thermodynamic quantities
    # ------------------------------------------------------------------ #
    rho  = U[0]
    vel  = U[1:4] / rho                # velocity vector  v = (u, v, w)
    E    = U[4]   / rho                # total energy per unit mass
    q2   = np.dot(vel, vel)            # |v|^2
    p    = (gamma - 1.0) * rho * (E - 0.5 * q2)
    c    = np.sqrt(gamma * p / rho)    # speed of sound
    H    = E + p / rho                 # total enthalpy per unit mass
    qn   = np.dot(vel, n)              # normal velocity  q_n
    M2   = q2 / c**2                   # Mach^2
    Mn   = qn / c                      # normal Mach number  M_n
    g1   = gamma - 1.0

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue qn  (eq. 3.6.24)
    # |qn| * (r2*l2' + r4*l4' + r5*l5')
    # ------------------------------------------------------------------ #
    mid = np.zeros((5, 5))

    # row 0
    mid[0, 0]   =  1.0 - 0.5 * g1 * M2
    mid[0, 1:4] =  (g1 / c**2) * vel
    mid[0, 4]   = -(g1 / c**2)

    # rows 1-3
    mid[1:4, 0]   = -0.5 * g1 * M2 * vel + qn * n
    mid[1:4, 1:4] =  (g1 / c**2) * np.outer(vel, vel) + np.eye(3) - np.outer(n, n)
    mid[1:4, 4]   = -(g1 / c**2) * vel

    # row 4
    mid[4, 0]   =  qn**2 - 0.5 * q2 * (1.0 + 0.5 * g1 * M2)
    mid[4, 1:4] =  (1.0 + 0.5 * g1 * M2) * vel - qn * n
    mid[4, 4]   = -0.5 * g1 * M2

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue (qn - c)  (eq. 3.6.25)
    # r1 * l1'
    # ------------------------------------------------------------------ #
    l1 = np.empty(5)
    l1[0]   =  0.25 * g1 * M2 + 0.5 * Mn
    l1[1:4] = -(g1 / (2.0 * c**2)) * vel - n / (2.0 * c)
    l1[4]   =  g1 / (2.0 * c**2)

    r1 = np.empty(5)
    r1[0]   =  1.0
    r1[1:4] =  vel - c * n
    r1[4]   =  H - qn * c

    A1 = np.outer(r1, l1)

    # ------------------------------------------------------------------ #
    # Contribution from eigenvalue (qn + c)  (eq. 3.6.26)
    # r3 * l3'
    # ------------------------------------------------------------------ #
    l3 = np.empty(5)
    l3[0]   =  0.25 * g1 * M2 - 0.5 * Mn
    l3[1:4] = -(g1 / (2.0 * c**2)) * vel + n / (2.0 * c)
    l3[4]   =  g1 / (2.0 * c**2)

    r3 = np.empty(5)
    r3[0]   =  1.0
    r3[1:4] =  vel + c * n
    r3[4]   =  H + qn * c

    A3 = np.outer(r3, l3)

    # ------------------------------------------------------------------ #
    # Assemble  |A_n| = |qn-c|*A1 + |qn|*mid + |qn+c|*A3   (eq. 3.6.19)
    # ------------------------------------------------------------------ #
    abs_An = abs(qn - c) * A1 + abs(qn) * mid + abs(qn + c) * A3

    return abs_An



def inviscid_flux_jacobians(U_i, U_j, n,gamma=1.4):
    """
    Compute the normal inviscid flux Jacobian blocks at a cell face between
    cell i (left/owner) and cell j (right/neighbor).

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E] of the
        two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz] of the face, pointing from i to j.
    gamma: float
        Ratio of specific heats
    Returns
    -------
    d(F)/d(U_i)  -- Inviscid Jacobian wrt the left-cell conservative state.
    d(F)/d(U_j)  -- Inviscid Jacobian wrt the right-cell conservative state.
    """

    U_roe = roe_average(U_i, U_j, gamma)
    abs_A_roe = absolute_normal_jacobian(U_roe, n, gamma=1.4)
    dFdUi = 0.5 * (compute_dFdU(U_i, n) + abs_A_roe)
    dFdUj = 0.5 * (compute_dFdU(U_j, n) - abs_A_roe)
    
    return dFdUi, dFdUj


# viscous jacobian function

"""
3D Normal Viscous Flux Jacobian (interface between two control volumes)
========================================================================

Implements d(F_n^v)/dU_i  and  d(F_n^v)/dU_j, i.e. the Jacobian of the
normal (projected) viscous flux F_n^v with respect to the conservative
variables of the two cells (i = "left", j = "right") sharing a face,
following the construction:

        F_n^v = [0, -tau_nx, -tau_ny, -tau_nz, -tau_nn + q_n]^T      (eq. 4.12.5)

        d F_n^v / dU = (d F_n^v / dW) * (dW / dU)                    (eq. 4.12.15)

with W = [rho, u, v, w, p]^T.

Modeling choices (consistent with the reference derivation):
  * Interface (averaged) primitive states are the simple arithmetic
    mean of the two cells:           phi_avg = (phi_i + phi_j) / 2   (eq. 4.12.25)
  * The normal derivative of any quantity is the directional
    finite difference across the two cell centers:
            d(phi)/dn = (phi_j - phi_i) / ds                          (eq. 4.12.26)
  * Because only the normal derivative is available (no compact
    stencil for the tangential derivatives), the full gradient is
    approximated as  grad(phi) ~= (d phi/dn) * n   (i.e. only the
    derivative along n is retained -- the same simplification used
    to reduce tau_bar.n to a function of d()/dn only, eq. 4.12.12-14).
  * Viscosity follows Sutherland's law (eq. 4.4.4) and its dependence
    on rho and p (through T = p/(rho*R)) is differentiated exactly
    via the symbolic chain rule -- this reproduces eq. (4.12.20)-(4.12.24)
    without having to hand-transcribe each partial derivative.

The two Jacobian blocks (wrt U_i and wrt U_j) are exactly what an
implicit (e.g. Newton/line-implicit/LU-SGS) viscous-flux assembly
needs for the off-diagonal/diagonal contributions of a face.

Author: generated to accompany the "3D Normal Viscous Flux and Jacobian"
notes (section 4.12).
"""

import numpy as np
import sympy as sp


# ----------------------------------------------------------------------
# Conservative -> primitive variables
# ----------------------------------------------------------------------
def cons_to_prim(U, gamma, Rgas):
    """
    U = [rho, rho*u, rho*v, rho*w, rho*E]
    returns rho, u, v, w, p, T  (numeric floats)
    """
    rho = U[0]
    u = U[1] / rho
    v = U[2] / rho
    w = U[3] / rho
    E = U[4] / rho
    p = (gamma - 1.0) * rho * (E - 0.5 * (u * u + v * v + w * w))
    T = p / (rho * Rgas)
    return rho, u, v, w, p, T


# ----------------------------------------------------------------------
# Sutherland's law (eq. 4.4.4), symbolic-friendly
# ----------------------------------------------------------------------
def sutherland_mu(T, mu0, T0, C):
    return mu0 * (T0 + C) / (T + C) * (T / T0) ** sp.Rational(3, 2)


# ----------------------------------------------------------------------
# dW/dU  (the right-hand matrix of eq. 4.12.15), evaluated at a given
# primitive state.  W = [rho, u, v, w, p]^T , U = [rho, rho u, rho v, rho w, rho E]^T
# ----------------------------------------------------------------------
def dWdU(rho, u, v, w, p, gamma):
    q2 = u * u + v * v + w * w
    M = np.array([
        [1.0,                 0.0,            0.0,            0.0,            0.0],
        [-u / rho,            1.0 / rho,       0.0,            0.0,            0.0],
        [-v / rho,            0.0,             1.0 / rho,      0.0,            0.0],
        [-w / rho,            0.0,             0.0,            1.0 / rho,      0.0],
        [0.5 * (gamma - 1.0) * q2, -(gamma - 1.0) * u, -(gamma - 1.0) * v, -(gamma - 1.0) * w, (gamma - 1.0)]
    ])
    return M


# ----------------------------------------------------------------------
# Symbolic construction of F_n^v as a function of the LEFT state
# (rho_L,u_L,v_L,w_L,p_L) and the RIGHT state (rho_R,u_R,v_R,w_R,p_R).
# Works for either state being symbolic; the other is plugged in as
# plain numbers.
# ----------------------------------------------------------------------
def _Fnv_symbolic(rho_L, u_L, v_L, w_L, p_L,
                   rho_R, u_R, v_R, w_R, p_R,
                   nx, ny, nz, ds,
                   gamma, Rgas, Pr, mu0, T0, Suth_C):

    T_L = p_L / (rho_L * Rgas)
    T_R = p_R / (rho_R * Rgas)

    # interface-averaged primitive state                       (eq. 4.12.25 style)
    rho_avg = (rho_L + rho_R) / 2
    u_avg   = (u_L   + u_R)   / 2
    v_avg   = (v_L   + v_R)   / 2
    w_avg   = (w_L   + w_R)   / 2
    T_avg   = (T_L   + T_R)   / 2

    mu_avg = sutherland_mu(T_avg, mu0, T0, Suth_C)             # mu(T_avg(rho,p)) -> exact chain rule via sympy

    # normal derivatives                                        (eq. 4.12.26)
    dudn = (u_R - u_L) / ds
    dvdn = (v_R - v_L) / ds
    dwdn = (w_R - w_L) / ds
    dTdn = (T_R - T_L) / ds

    # full gradient approximated by the normal derivative only:
    #   grad(phi) ~= (d phi/dn) * n   =>   d phi/dx_k = (d phi/dn) * n_k
    div = dudn * nx + dvdn * ny + dwdn * nz   # du/dx + dv/dy + dw/dz

    # Newtonian viscous stress tensor with Stokes' hypothesis      (eq. 4.12.16-19)
    tau_xx = mu_avg * (2 * dudn * nx - sp.Rational(2, 3) * div)
    tau_yy = mu_avg * (2 * dvdn * ny - sp.Rational(2, 3) * div)
    tau_zz = mu_avg * (2 * dwdn * nz - sp.Rational(2, 3) * div)
    tau_xy = mu_avg * (dudn * ny + dvdn * nx)
    tau_xz = mu_avg * (dudn * nz + dwdn * nx)
    tau_yz = mu_avg * (dvdn * nz + dwdn * ny)

    # projection along n                                          (eq. 4.12.6-4.12.9)
    tau_nx = tau_xx * nx + tau_xy * ny + tau_xz * nz
    tau_ny = tau_xy * nx + tau_yy * ny + tau_yz * nz
    tau_nz = tau_xz * nx + tau_yz * ny + tau_zz * nz
    tau_nn = tau_nx * u_avg + tau_ny * v_avg + tau_nz * w_avg

    # heat flux                                                    (eq. 4.12.11)
    kappa_avg = gamma * mu_avg / (Pr * (gamma - 1.0))
    q_n = -kappa_avg * dTdn

    F0 = sp.Integer(0)
    F1 = -tau_nx
    F2 = -tau_ny
    F3 = -tau_nz
    F4 = -tau_nn + q_n
    return sp.Matrix([F0, F1, F2, F3, F4])


# ----------------------------------------------------------------------
# Purely numeric evaluation of F_n^v(U_i, U_j, n, ds)  -- the discretized
# normal viscous flux function itself (no differentiation). This is the
# "flux(U)" used by the finite-difference Jacobian below, and it uses
# exactly the same modeling choices as the analytic version above:
#   * interface state = arithmetic mean of the two cells   (eq. 4.12.25)
#   * spatial (normal) derivative = directional finite difference
#         d(phi)/dn = (phi_j - phi_i) / ds                  (eq. 4.12.26)
#     i.e. the ONLY spatial derivative information available is the
#     one-sided difference of the two cell-center values along the
#     line joining them; the gradient is then assumed aligned with n:
#         grad(phi) ~= (d phi/dn) * n
#     (tangential derivatives are not reconstructed/are neglected).
# ----------------------------------------------------------------------

# ----------------------------------------------------------------------
# Main entry point
# ----------------------------------------------------------------------
def viscous_flux_jacobians(U_i, U_j, n, ds,
                            gamma=1.4, Rgas=287.0, Pr=0.72,
                            mu0=1.716e-5, T0=273.15, Suth_C=110.4):
    """
    Compute the normal viscous flux Jacobian blocks at a cell face between
    cell i (left/owner) and cell j (right/neighbor).

    Parameters
    ----------
    U_i, U_j : array_like, shape (5,)
        Conservative variables [rho, rho*u, rho*v, rho*w, rho*E] of the
        two cells sharing the face.
    n : array_like, shape (3,)
        Unit normal vector [nx, ny, nz] of the face, pointing from i to j.
    ds : float
        Distance between the two cell centers (used for the directional
        finite-difference normal derivative, eq. 4.12.26).
    gamma, Rgas, Pr : float
        Ratio of specific heats, specific gas constant, Prandtl number.
    mu0, T0, Suth_C : float
        Sutherland's law reference viscosity, reference temperature and
        Sutherland constant (eq. 4.4.4).

    Returns
    -------
    J_i : ndarray, shape (5,5)
        d(F_n^v)/d(U_i)  -- Jacobian wrt the left-cell conservative state.
    J_j : ndarray, shape (5,5)
        d(F_n^v)/d(U_j)  -- Jacobian wrt the right-cell conservative state.
    """
    U_i = np.asarray(U_i, dtype=float)
    U_j = np.asarray(U_j, dtype=float)
    nx, ny, nz = n

    rho_i, u_i, v_i, w_i, p_i, _ = cons_to_prim(U_i, gamma, Rgas)
    rho_j, u_j, v_j, w_j, p_j, _ = cons_to_prim(U_j, gamma, Rgas)

    rho_s, u_s, v_s, w_s, p_s = sp.symbols('rho_s u_s v_s w_s p_s')
    Wsyms = (rho_s, u_s, v_s, w_s, p_s)

    # ---- Jacobian wrt the LEFT state (U_i) ----
    Fnv_i = _Fnv_symbolic(rho_s, u_s, v_s, w_s, p_s,
                           rho_j, u_j, v_j, w_j, p_j,
                           nx, ny, nz, ds, gamma, Rgas, Pr, mu0, T0, Suth_C)
    dFdW_i = Fnv_i.jacobian(Wsyms)
    dFdW_i_num = np.array(
        dFdW_i.subs({rho_s: rho_i, u_s: u_i, v_s: v_i, w_s: w_i, p_s: p_i})
    ).astype(np.float64)
    J_i = dFdW_i_num @ dWdU(rho_i, u_i, v_i, w_i, p_i, gamma)

    # ---- Jacobian wrt the RIGHT state (U_j) ----
    Fnv_j = _Fnv_symbolic(rho_i, u_i, v_i, w_i, p_i,
                           rho_s, u_s, v_s, w_s, p_s,
                           nx, ny, nz, ds, gamma, Rgas, Pr, mu0, T0, Suth_C)
    dFdW_j = Fnv_j.jacobian(Wsyms)
    dFdW_j_num = np.array(
        dFdW_j.subs({rho_s: rho_j, u_s: u_j, v_s: v_j, w_s: w_j, p_s: p_j})
    ).astype(np.float64)
    J_j = dFdW_j_num @ dWdU(rho_j, u_j, v_j, w_j, p_j, gamma)

    return J_i, J_j




In [9]:
def boundary_flux_jacobian(U_i, bc_type, n, ds=1.0,
                            gamma=1.4, R_gas=287.0, Pr=0.72,
                            mu0=1.716e-5, T0=273.15, Suth_C=110.4,
                            u_b=0.0, v_b=0.0, w_b=0.0, T_b=300.0, P_b=101325.0):
    """
    Compute the boundary flux Jacobian dF_bc/dU_i at a boundary face.

    The ghost cell U_g is constructed from the interior state U_i via
    ghost_state(), then the appropriate flux Jacobian is evaluated using
    (U_i, U_g) as the left/right pair. Only the left (interior) block
    dF/dU_i is returned; the ghost-cell dependence on U_i is frozen
    (standard first-order boundary Jacobian treatment).

    Parameters
    ----------
    U_i     : array (5,)    interior conservative state
    bc_type : str           'slip', 'riemann' --> inviscid Jacobian
                            'noslip'          --> viscous Jacobian (inviscid neglected)
    n       : array (3,)    outward unit normal
    ds      : float         cell-center to face distance (needed for 'noslip' only)

    For bc_type == 'riemann':
    u_b, v_b, w_b, T_b, P_b : float  prescribed boundary state

    Returns
    -------
    J_bc : ndarray (5, 5)   dF_bc / dU_i
    """
    U_i = np.asarray(U_i, dtype=float)
    n   = np.asarray(n,   dtype=float)
    n   = n / np.linalg.norm(n)

    # ------------------------------------------------------------------ #
    # Build ghost cell
    # ------------------------------------------------------------------ #
    if bc_type == 'riemann':
        U_g = ghost_state(U_i, 'riemann', n, gamma, R_gas,
                          u_b, v_b, w_b, T_b, P_b)
    else:
        U_g = ghost_state(U_i, bc_type, n)

    # ------------------------------------------------------------------ #
    # Select flux Jacobian
    # ------------------------------------------------------------------ #
    if bc_type in ('slip', 'riemann'):
        # inviscid (Roe-based) Jacobian; return only the interior block
        J_bc, _ = inviscid_flux_jacobians(U_i, U_g, n, gamma)

    elif bc_type == 'noslip':
        # viscous Jacobian only (inviscid flux is zero at a no-slip wall)
        J_bc, _ = viscous_flux_jacobians(U_i, U_g, n, ds,
                                          gamma, R_gas, Pr, mu0, T0, Suth_C)
    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'. "
                         "Use 'slip', 'noslip', or 'riemann'.")

    return J_bc

modify boundary flux jacobian
- since the slip wall(front surface) is very close to our region of interest
- only very small number of nodes are truely interior since our domain size is so small
- applying to slip wall/riemann boundary conditions only due to simplicity (no-slip wall is far away from region of interst)
- to make it account for non-frozen ghost state
- U_ghost = ghost_state(U_i, bc, ...), the ghost_node is a function of Ui

F_bc = 0.5 * (F(Ui) + F(Uj)) +  0.5 *abs_tidle_A *(Ui-Ug), function of Ui and Ug

Frozen ghost node = Treat U_ghost as a constant (independent of Ui)

Such that: ∂F_bc/∂U_i =  0.5 * (dF/dU(Ui)  + abs_tidle_A)   ← frozen: ∂U_ghost/∂U_i = 0 

But the true derivative: ∂F_bc/∂U_i = 0.5 * (dF/dU(Ui) + dF/dU(Ug) * dUg/dUi) + 0.5 *abs_tidle_A * (1 -dUg/dUi)   ← full chain rule

∂F_bc/∂U_i  = [0.5*dF/dU(U_i) + 0.5*|A|]  +  [0.5*dF/dU(U_g) - 0.5*|A|] @ dUg/dUi
            =  J_L +J_R @ dUg/dUi
Required function
- ghost_state_jacobian = ∂U_ghost/∂U_i 

In [ ]:
# ── JAX-compatible ghost state ────────────────────────────────────────────────
def ghost_state_jax(U_i, bc, n_bc, gamma=1.4, R_gas=287.0,
                    u_b=0., v_b=0., w_b=0., T_b=288.15, P_b=101325.):
    """
    JAX-traceable mirror of ghost_state / riemann_invariant_bc.
    All Python if/else on bc type are static (compile-time).
    All branches on flow quantities use jnp.where (trace-time).
    """
    # normalise (matches your np.linalg.norm step)
    n_jax = jnp.array(n_bc, dtype=jnp.float64)
    n_jax = n_jax / jnp.linalg.norm(n_jax)
    nx, ny, nz = n_jax[0], n_jax[1], n_jax[2]

    if bc == 'slip':
        rho = U_i[0]
        vel = U_i[1:4] / rho
        vn  = vel[0]*nx + vel[1]*ny + vel[2]*nz
        vg  = vel - 2.0*vn*n_jax
        return jnp.stack([rho, rho*vg[0], rho*vg[1], rho*vg[2], U_i[4]])

    elif bc == 'noslip':
        return jnp.stack([U_i[0], -U_i[1], -U_i[2], -U_i[3], U_i[4]])

    elif bc == 'riemann':
        cv = R_gas / (gamma - 1.0)

        # ── interior primitives ──────────────────────────────────────────
        rho_i  = U_i[0]
        vel_i  = U_i[1:4] / rho_i
        Vn_i   = vel_i[0]*nx + vel_i[1]*ny + vel_i[2]*nz
        Vt_i   = vel_i - Vn_i * n_jax
        p_i    = (gamma - 1.0) * (U_i[4]
                   - 0.5 * rho_i * jnp.dot(vel_i, vel_i))
        T_i    = p_i / (rho_i * R_gas)
        c_i    = jnp.sqrt(jnp.clip(gamma * R_gas * T_i, 1e-10, None))
        Mn_i   = Vn_i / c_i
        s_i    = p_i / rho_i**gamma

        # ── prescribed BC (all constants — zero gradient through AD) ─────
        rho_b  = float(P_b / (R_gas * T_b))
        c_b    = float(np.sqrt(gamma * R_gas * T_b))
        Vn_b   = float(u_b*float(nx) + v_b*float(ny) + w_b*float(nz))
        vel_b  = jnp.array([u_b, v_b, w_b], dtype=jnp.float64)
        Vt_b   = vel_b - Vn_b * n_jax
        s_b    = float(P_b / rho_b**gamma)

        # ── Riemann invariants ───────────────────────────────────────────
        fac     = 2.0 / (gamma - 1.0)
        Rp_int  = Vn_i + fac * c_i               # outgoing from interior
        Rm_b    = float(Vn_b - fac * c_b)        # incoming from BC (constant)

        Vn_rim  = 0.5 * (Rp_int + Rm_b)
        c_rim   = jnp.clip(0.25*(gamma - 1.0)*(Rp_int - Rm_b), 1e-10, None)

        # ── select by regime (all jnp.where — fully traceable) ───────────
        is_sup_in  = Mn_i <= -1.0          # supersonic inflow
        is_sup_out = Mn_i >=  1.0          # supersonic outflow
        is_inflow  = is_sup_in | (Vn_i < 0.0)   # sup-inflow OR sub-inflow

        Vn_wall = jnp.where(is_sup_in,  float(Vn_b),
                  jnp.where(is_sup_out, Vn_i, Vn_rim))

        c_wall  = jnp.where(is_sup_in,  float(c_b),
                  jnp.where(is_sup_out, c_i,  c_rim))

        Vt_wall = jnp.where(is_inflow, Vt_b, Vt_i)   # (3,) — broadcasts
        s_wall  = jnp.where(is_inflow, float(s_b), s_i)

        # ── reconstruct wall primitive state ─────────────────────────────
        rho_wall = (c_wall**2 / (gamma * s_wall))**(1.0 / (gamma - 1.0))
        p_wall   = s_wall * rho_wall**gamma
        T_wall   = p_wall / (rho_wall * R_gas)
        vel_wall = Vn_wall * n_jax + Vt_wall
        E_wall   = cv * T_wall + 0.5 * jnp.dot(vel_wall, vel_wall)

        U_wall = jnp.stack([
            rho_wall,
            rho_wall * vel_wall[0],
            rho_wall * vel_wall[1],
            rho_wall * vel_wall[2],
            rho_wall * E_wall,
        ])

        # ── ghost = mirror wall through interior  (matches your code) ────
        return 2.0 * U_wall - U_i

    else:
        raise ValueError(f"bc='{bc}' not supported")


# ── dUg/dUi via JAX forward-mode AD ──────────────────────────────────────────
def ghost_state_jacobian_ad(U_i, bc, n_bc, gamma=1.4, R_gas=287.0,
                             u_b=0., v_b=0., w_b=0., T_b=288.15, P_b=101325.):
    """
    Returns dUg/dUi as a (5, 5) numpy array via JAX jacfwd.

    jacfwd computes the Jacobian column by column using forward-mode AD
    (5 JVPs for 5 input dimensions) — efficient for a (5→5) function.
    """
    U_i_jax = jnp.array(U_i, dtype=jnp.float64)

    fn = lambda U: ghost_state_jax(U, bc, n_bc, gamma, R_gas,
                                   u_b, v_b, w_b, T_b, P_b)

    dUg_dUi = jax.jacfwd(fn)(U_i_jax)   # (5, 5)
    return np.array(dUg_dUi)


# ── FD fallback — cross-check against autodiff ───────────────────────────────
def ghost_state_jacobian_fd(U_i, bc, n_bc, gamma=1.4, R_gas=287.0,
                             u_b=0., v_b=0., w_b=0., T_b=288.15, P_b=101325.,
                             eps=1e-6):
    """
    Finite-difference dUg/dUi — use to verify ghost_state_jacobian_ad.
    Uses the original (numpy) ghost_state function directly.
    """
    dUg = np.zeros((5, 5))
    U_g0 = ghost_state(U_i, bc, n_bc, gamma, R_gas, u_b, v_b, w_b, T_b, P_b)
    for k in range(5):
        ej       = np.zeros(5); ej[k] = eps
        U_g_fwd  = ghost_state(U_i + ej, bc, n_bc, gamma, R_gas,
                               u_b, v_b, w_b, T_b, P_b)
        dUg[:, k] = (U_g_fwd - U_g0) / eps
    return dUg


# ── Quick verification: AD vs FD ──────────────────────────────────────────────
def verify_ghost_jacobian(U_i, bc, n_bc, gamma=1.4, R_gas=287.0,
                           u_b=0., v_b=0., w_b=0., T_b=288.15, P_b=101325.):

    dUg_ad = ghost_state_jacobian_ad(U_i, bc, n_bc, gamma, R_gas,
                                     u_b, v_b, w_b, T_b, P_b)
    dUg_fd = ghost_state_jacobian_fd(U_i, bc, n_bc, gamma, R_gas,
                                     u_b, v_b, w_b, T_b, P_b)

    err = np.linalg.norm(dUg_ad - dUg_fd) / (np.linalg.norm(dUg_fd) + 1e-14)
    print(f"BC='{bc}'  dUg/dUi  AD vs FD  rel_err = {err:.4e}")
    print(f"  dUg/dUi (AD):\n{np.round(dUg_ad, 6)}")
    return dUg_ad, dUg_fd


In [11]:
# verification with aribitary test case

# ── Arbitrary test state (subsonic, Ma ≈ 0.2) ─────────────────────────────────
gamma  = 1.4
R_gas  = 287.0

rho    = 1.2
u, v, w = 50.0, 10.0, 0.0
p      = 101325.0
E      = p / ((gamma-1)*rho) + 0.5*(u**2 + v**2 + w**2)

U_test = np.array([rho, rho*u, rho*v, rho*w, rho*E])

# BC freestream (Riemann)
T_in = 288.15
P_in = 101325.0
u_in, v_in, w_in = 68.0, 0.0, 0.0   # Ma ≈ 0.2

# ── Test normals ───────────────────────────────────────────────────────────────
normals = {
    'x-face'  : np.array([1., 0., 0.]),
    'y-face'  : np.array([0., 1., 0.]),
    'diagonal': np.array([1., 1., 0.]) / np.sqrt(2),
}

# ── Run ───────────────────────────────────────────────────────────────────────
for n_name, n_test in normals.items():
    print(f"\n{'='*60}")
    print(f"Normal: {n_name}  {n_test}")

    for bc in ('slip', 'riemann'):
        kw = dict(gamma=gamma, R_gas=R_gas)
        if bc == 'riemann':
            kw.update(u_b=u_in, v_b=v_in, w_b=w_in, T_b=T_in, P_b=P_in)

        dUg_ad, dUg_fd = verify_ghost_jacobian(U_test, bc, n_test, **kw)
        print()


Normal: x-face  [1. 0. 0.]
BC='slip'  dUg/dUi  AD vs FD  rel_err = 3.4053e-06
  dUg/dUi (AD):
[[ 1.  0.  0.  0.  0.]
 [-0. -1.  0.  0.  0.]
 [ 0.  0.  1.  0.  0.]
 [ 0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  1.]]

BC='riemann'  dUg/dUi  AD vs FD  rel_err = 5.8358e-07
  dUg/dUi (AD):
[[ 3.09174100e+00  3.34500000e-03  1.11000000e-04  0.00000000e+00
  -1.10000000e-05]
 [-5.80357575e+02 -1.70610000e-02 -3.10930000e-02  0.00000000e+00
   3.10900000e-03]
 [ 2.19368820e+01  3.34530000e-02  8.99165000e-01  0.00000000e+00
  -1.11000000e-04]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  8.98052000e-01
   0.00000000e+00]
 [ 5.88474292e+05  9.34430523e+02  3.02171650e+01  0.00000000e+00
  -2.12366400e+00]]


Normal: y-face  [0. 1. 0.]
BC='slip'  dUg/dUi  AD vs FD  rel_err = 3.4053e-06
  dUg/dUi (AD):
[[ 1.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.]
 [-0.  0. -1.  0.  0.]
 [ 0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  1.]]

BC='riemann'  dUg/dUi  AD vs FD  rel_err = 6.1202e-07
  dUg/dUi (AD):
[[ 3.3996

Modified boundary_flux_jacobian with non-frozen_ghost node capability

In [12]:
# ── JAX-compatible ghost state ────────────────────────────────────────────────
def ghost_state_jax(U_i, bc, n_bc, gamma=1.4, R_gas=287.0,
                    u_b=0., v_b=0., w_b=0., T_b=288.15, P_b=101325.):
    """
    JAX-traceable mirror of ghost_state / riemann_invariant_bc.
    All Python if/else on bc type are static (compile-time).
    All branches on flow quantities use jnp.where (trace-time).
    """
    # normalise (matches your np.linalg.norm step)
    n_jax = jnp.array(n_bc, dtype=jnp.float64)
    n_jax = n_jax / jnp.linalg.norm(n_jax)
    nx, ny, nz = n_jax[0], n_jax[1], n_jax[2]

    if bc == 'slip':
        rho = U_i[0]
        vel = U_i[1:4] / rho
        vn  = vel[0]*nx + vel[1]*ny + vel[2]*nz
        vg  = vel - 2.0*vn*n_jax
        return jnp.stack([rho, rho*vg[0], rho*vg[1], rho*vg[2], U_i[4]])

    elif bc == 'noslip':
        return jnp.stack([U_i[0], -U_i[1], -U_i[2], -U_i[3], U_i[4]])

    elif bc == 'riemann':
        cv = R_gas / (gamma - 1.0)

        # ── interior primitives ──────────────────────────────────────────
        rho_i  = U_i[0]
        vel_i  = U_i[1:4] / rho_i
        Vn_i   = vel_i[0]*nx + vel_i[1]*ny + vel_i[2]*nz
        Vt_i   = vel_i - Vn_i * n_jax
        p_i    = (gamma - 1.0) * (U_i[4]
                   - 0.5 * rho_i * jnp.dot(vel_i, vel_i))
        T_i    = p_i / (rho_i * R_gas)
        c_i    = jnp.sqrt(jnp.clip(gamma * R_gas * T_i, 1e-10, None))
        Mn_i   = Vn_i / c_i
        s_i    = p_i / rho_i**gamma

        # ── prescribed BC (all constants — zero gradient through AD) ─────
        rho_b  = float(P_b / (R_gas * T_b))
        c_b    = float(np.sqrt(gamma * R_gas * T_b))
        Vn_b   = float(u_b*float(nx) + v_b*float(ny) + w_b*float(nz))
        vel_b  = jnp.array([u_b, v_b, w_b], dtype=jnp.float64)
        Vt_b   = vel_b - Vn_b * n_jax
        s_b    = float(P_b / rho_b**gamma)

        # ── Riemann invariants ───────────────────────────────────────────
        fac     = 2.0 / (gamma - 1.0)
        Rp_int  = Vn_i + fac * c_i               # outgoing from interior
        Rm_b    = float(Vn_b - fac * c_b)        # incoming from BC (constant)

        Vn_rim  = 0.5 * (Rp_int + Rm_b)
        c_rim   = jnp.clip(0.25*(gamma - 1.0)*(Rp_int - Rm_b), 1e-10, None)

        # ── select by regime (all jnp.where — fully traceable) ───────────
        is_sup_in  = Mn_i <= -1.0          # supersonic inflow
        is_sup_out = Mn_i >=  1.0          # supersonic outflow
        is_inflow  = is_sup_in | (Vn_i < 0.0)   # sup-inflow OR sub-inflow

        Vn_wall = jnp.where(is_sup_in,  float(Vn_b),
                  jnp.where(is_sup_out, Vn_i, Vn_rim))

        c_wall  = jnp.where(is_sup_in,  float(c_b),
                  jnp.where(is_sup_out, c_i,  c_rim))

        Vt_wall = jnp.where(is_inflow, Vt_b, Vt_i)   # (3,) — broadcasts
        s_wall  = jnp.where(is_inflow, float(s_b), s_i)

        # ── reconstruct wall primitive state ─────────────────────────────
        rho_wall = (c_wall**2 / (gamma * s_wall))**(1.0 / (gamma - 1.0))
        p_wall   = s_wall * rho_wall**gamma
        T_wall   = p_wall / (rho_wall * R_gas)
        vel_wall = Vn_wall * n_jax + Vt_wall
        E_wall   = cv * T_wall + 0.5 * jnp.dot(vel_wall, vel_wall)

        U_wall = jnp.stack([
            rho_wall,
            rho_wall * vel_wall[0],
            rho_wall * vel_wall[1],
            rho_wall * vel_wall[2],
            rho_wall * E_wall,
        ])

        # ── ghost = mirror wall through interior  (matches your code) ────
        return 2.0 * U_wall - U_i

    else:
        raise ValueError(f"bc='{bc}' not supported")


# ── dUg/dUi via JAX forward-mode AD ──────────────────────────────────────────
def ghost_state_jacobian_ad(U_i, bc, n_bc, gamma=1.4, R_gas=287.0,
                             u_b=0., v_b=0., w_b=0., T_b=288.15, P_b=101325.):
    """
    Returns dUg/dUi as a (5, 5) numpy array via JAX jacfwd.

    jacfwd computes the Jacobian column by column using forward-mode AD
    (5 JVPs for 5 input dimensions) — efficient for a (5→5) function.
    """
    U_i_jax = jnp.array(U_i, dtype=jnp.float64)

    fn = lambda U: ghost_state_jax(U, bc, n_bc, gamma, R_gas,
                                   u_b, v_b, w_b, T_b, P_b)

    dUg_dUi = jax.jacfwd(fn)(U_i_jax)   # (5, 5)
    return np.array(dUg_dUi)


def boundary_flux_jacobian(U_i, bc_type, n, ds=1.0,
                            gamma=1.4, R_gas=287.0, Pr=0.72,
                            mu0=1.716e-5, T0=273.15, Suth_C=110.4,
                            u_b=0.0, v_b=0.0, w_b=0.0, T_b=300.0, P_b=101325.0,
                            frozen_ghost=False):
    """
    Compute the boundary flux Jacobian dF_bc/dU_i at a boundary face.

    Parameters
    ----------
    frozen_ghost : bool
        If False (default), accounts for dU_ghost/dU_i via JAX autodiff —
        exact chain-rule Jacobian for 'slip' and 'riemann'.
        If True, reverts to frozen-ghost (original first-order approximation).

    Returns
    -------
    J_bc : ndarray (5, 5)   dF_bc / dU_i
    """
    U_i = np.asarray(U_i, dtype=float)
    n   = np.asarray(n,   dtype=float)
    n   = n / np.linalg.norm(n)

    # ── Build ghost cell ───────────────────────────────────────────────────
    if bc_type == 'riemann':
        U_g = ghost_state(U_i, 'riemann', n, gamma, R_gas,
                          u_b, v_b, w_b, T_b, P_b)
    else:
        U_g = ghost_state(U_i, bc_type, n)

    # ── Inviscid BCs (slip / riemann) ──────────────────────────────────────
    if bc_type in ('slip', 'riemann'):
        # J_L = A+  (interior block)
        # J_R = A-  (ghost block)  — both already computed by inviscid_flux_jacobians
        J_L, J_R = inviscid_flux_jacobians(U_i, U_g, n, gamma)

        if frozen_ghost:
            # original frozen approximation: dF/dU_i ≈ A+
            return J_L

        # exact:  dF_bc/dU_i = A+ + A- @ dUg/dUi
        kw = dict(gamma=gamma, R_gas=R_gas)
        if bc_type == 'riemann':
            kw.update(u_b=u_b, v_b=v_b, w_b=w_b, T_b=T_b, P_b=P_b)

        dUg_dUi = ghost_state_jacobian_ad(U_i, bc_type, n, **kw)
        return J_L + J_R @ dUg_dUi

    # ── Viscous BC (noslip) ────────────────────────────────────────────────
    elif bc_type == 'noslip':
        # noslip ghost: U_g = [rho, -rho*u, -rho*v, -rho*w, rho*E]
        # so dUg/dUi = diag([1, -1, -1, -1, 1]) — but viscous Jacobian
        # treatment is left as frozen for now (viscous term dominates near wall)
        J_bc, _ = viscous_flux_jacobians(U_i, U_g, n, ds,
                                          gamma, R_gas, Pr, mu0, T0, Suth_C)
        return J_bc

    else:
        raise ValueError(f"Unknown bc_type '{bc_type}'. "
                         "Use 'slip', 'noslip', or 'riemann'.")

In [ ]:
# verification with aribitary test case

# ── Arbitrary test state (subsonic, Ma ≈ 0.2) ─────────────────────────────────
gamma  = 1.4
R_gas  = 287.0

rho    = 1.2
u, v, w = 50.0, 10.0, 0.0
p      = 101325.0
E      = p / ((gamma-1)*rho) + 0.5*(u**2 + v**2 + w**2)

U_test = np.array([rho, rho*u, rho*v, rho*w, rho*E])

# BC freestream (Riemann)
T_in = 288.15
P_in = 101325.0
u_in, v_in, w_in = 68.0, 0.0, 0.0   # Ma ≈ 0.2

# ── Test normals ───────────────────────────────────────────────────────────────
normals = {
    'x-face'  : np.array([1., 0., 0.]),
    'y-face'  : np.array([0., 1., 0.]),
    'diagonal': np.array([1., 1., 0.]) / np.sqrt(2),
}

# ── Run ───────────────────────────────────────────────────────────────────────
for n_name, n_test in normals.items():
    print(f"\n{'='*60}")
    print(f"Normal: {n_name}  {n_test}")

    for bc in ('slip', 'riemann'):
        kw = dict(gamma=gamma, R_gas=R_gas)
        if bc == 'riemann':
            kw.update(u_b=u_in, v_b=v_in, w_b=w_in, T_b=T_in, P_b=P_in)

        J_frozen = boundary_flux_jacobian(U_test, 'riemann', n_test, frozen_ghost=True,
                                   u_b=u_in, v_b=v_in, w_b=w_in, T_b=T_in, P_b=P_in,
                                   gamma=gamma, R_gas=R_gas)
        J_exact  = boundary_flux_jacobian(U_test, 'riemann', n_test frozen_ghost=False,
                                          u_b=u_in, v_b=v_in, w_b=w_in, T_b=T_in, P_b=P_in,
                                          gamma=gamma, R_gas=R_gas)
        print("Correction norm ||J_exact - J_frozen|| / ||J_frozen||:",
      np.linalg.norm(J_exact - J_frozen) / np.linalg.norm(J_frozen))



Normal: x-face  [1. 0. 0.]


NameError: name 'n' is not defined

Verification by checking the analytical forms of boundary flux jacobians
1. Slip-wall boundary flux jacobian
- dF/dU|wall = dF/dV|wall * dV/dU

2. Subsonic inlet flux Jacobian

3. Subsonic outlet flux jacobian


In [ ]:
# analytical form of Slip-wall boundary flux jacobian